In [10]:
import os.path
import gc
import pandas as pd
import pickle as pkl
import numpy as np
import random
from tqdm import tqdm
import json
from sklearn import metrics
from utils import get_texts_for_ptype
from sklearn import metrics
from scipy.optimize import minimize_scalar
from custom_models.custom_model import BatchVectorizerClassifier
from ray import tune

In [3]:
model_types = [
    "hinge", 
    "log_loss", 
    "modified_huber", 
    "squared_hinge", 
    "perceptron"]
neg_to_pos_ratio = 1

results_file = "results_titles_abstracts_linear_features1000.json"

inputs = "titles_abstracts"

filename_pos_human = "../data/pubmed_articles_pos_human.csv"
filename_neg_human = "../data/pubmed_articles_neg_human.csv"

with open("../data/cleaned_pos_neg_train_val_test_dataset_ids.pkl", "rb") as f:
    prop_typ_id_dict = pkl.load(f)

ptypes = prop_typ_id_dict.keys()

In [4]:
with open(results_file, "r") as f_read:
    results_list = json.load(f_read)

In [5]:
# decision_fn_file = "results_decision_fn_linear_features1000.json"

In [6]:
def predict(df, db=0):
    return [int(x>=db) for x in df ]

In [7]:
neg_pos_ratio_dict = {
    "is_rct": 5585042/489295,
    "is_sr": 5659292/121063,
    "is_casecontrol": 5628219/65790,
    "is_cohort": 5632460/78688,
    "is_guideline": 5676411/33826
}

def calc_precision(recall, specificity, neg_pos_ratio):
    return recall / (recall + neg_pos_ratio * (1 - specificity) + 1e-15)

def calc_f1(recall, precision):
    return 2*recall*precision/(recall+precision + 1e-15)

In [8]:
def find_db(db):
    y_pred_test = predict(df, db)
    recall_test = metrics.recall_score(y_test, y_pred_test, pos_label=1)
    return abs(recall_test - 0.985)

In [11]:
metrics_lst = []
for results_json in tqdm(results_list):
    ptype = results_json["ptype"]
    model_type = results_json["model_type"]
    X_train, X_val, X_test, y_train, y_val, y_test = get_texts_for_ptype(ptype, inputs="titles_abstracts", neg_to_pos_ratio=neg_to_pos_ratio)    
    bvc = BatchVectorizerClassifier(
                    loss=results_json["model_type"],
                    penalty=results_json["best_penalty"],
                    alpha=results_json["best_alpha"],
                    l1_ratio=0.15,
                    fit_intercept=True,
                    max_iter=1000,
                    tol=0.001,
                    shuffle=True,
                    verbose=0,
                    epsilon=0.1,
                    n_jobs=None,
                    random_state=None,
                    learning_rate="optimal",
                    eta0=0.0,
                    power_t=0.5,
                    early_stopping=False,
                    validation_fraction=0.1,
                    n_iter_no_change=5,
                    class_weight=None,
                    warm_start=False,
                    average=False,
        )
    bvc.fit(X_train, y_train)
    df = bvc.decision_function(X_test)
    
    auc_roc = metrics.roc_auc_score(y_test, df)
    
    res = minimize_scalar(find_db)
    db = res["x"]
    y_pred_test = predict(df, db)
    
    recall_test = metrics.recall_score(y_test, y_pred_test, pos_label=1)
    spec_test = metrics.recall_score(y_test, y_pred_test, pos_label=0)
    precision_test = calc_precision(recall_test, spec_test, neg_pos_ratio_dict[ptype])
    f1_test = calc_f1(recall_test, precision_test)
    
    metrics_dict = {
        "ptype": ptype,
        "model_type": model_type,
        "y_test": y_test,
        "db": db,
        "df": df,
        "auc_roc": auc_roc,
        "recall_test": recall_test,
        "spec_test": spec_test,
        "precision_test": precision_test,
        "f1_test": f1_test
    }
    metrics_lst.append(metrics_dict)
    
    if os.path.exists(gbdt_metrics_adjusted_results_path):
        with open(gbdt_metrics_adjusted_results_path, "rb") as f:
            metrics_lst = pkl.load(f)
    
    metrics_lst.append(metrics_dict)
    with open(gbdt_metrics_adjusted_results_path, "wb") as f:
        pkl.dump(metrics_lst, f)
        
    del bvc
    gc.collect()

  0%|                                                    | 0/25 [00:00<?, ?it/s]
0it [00:00, ?it/s]
1it [00:01,  1.34s/it]
2it [00:02,  1.37s/it]
3it [00:04,  1.32s/it]
4it [00:05,  1.31s/it]
5it [00:06,  1.34s/it]
6it [00:08,  1.34s/it]
7it [00:09,  1.36s/it]
8it [00:10,  1.34s/it]
9it [00:12,  1.34s/it]
10it [00:13,  1.36s/it]
11it [00:14,  1.32s/it]
12it [00:15,  1.31s/it]
13it [00:17,  1.33s/it]
14it [00:18,  1.35s/it]
15it [00:20,  1.38s/it]
16it [00:21,  1.39s/it]
17it [00:22,  1.35s/it]
18it [00:24,  1.35s/it]
19it [00:25,  1.38s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:

20it [00:27,  1.39s/it]
21it [00:28,  1.37s/it]
22it [00:29,  1.39s/it]
23it [00:31,  1.37s/it]
24it [00:32,  1.34s/it]
25it [00:33,  1.34s/it]
26it [00:35,  1.36s/it]
27it [00:36,  1.36s/it]
28it [00:37,  1.35s/it]
29it [00:39,  1.32s/it]
30it [00:40,  1.33s/i

39it [00:59,  1.52s/it]
40it [01:00,  1.50s/it]
41it [01:02,  1.50s/it]
42it [01:03,  1.48s/it]
43it [01:05,  1.52s/it]
44it [01:06,  1.54s/it]
45it [01:08,  1.53s/it]
46it [01:10,  1.55s/it]
47it [01:11,  1.56s/it]
48it [01:13,  1.55s/it]
49it [01:14,  1.51s/it]
50it [01:15,  1.48s/it]
51it [01:17,  1.50s/it]
52it [01:19,  1.52s/it]
53it [01:20,  1.51s/it]
54it [01:22,  1.50s/it]
55it [01:23,  1.56s/it]
56it [01:25,  1.52s/it]
57it [01:26,  1.51s/it]
  8%|███▎                                     | 2/25 [06:49<1:18:24, 204.53s/it]
0it [00:00, ?it/s]
1it [00:01,  1.33s/it]
2it [00:02,  1.36s/it]
3it [00:03,  1.31s/it]
4it [00:05,  1.30s/it]
5it [00:06,  1.33s/it]
6it [00:07,  1.33s/it]
7it [00:09,  1.35s/it]
8it [00:10,  1.33s/it]
9it [00:11,  1.32s/it]
10it [00:13,  1.34s/it]
11it [00:14,  1.31s/it]
12it [00:15,  1.30s/it]
13it [00:17,  1.32s/it]
14it [00:18,  1.34s/it]
15it [00:19,  1.36s/it]
16it [00:21,  1.38s/it]
17it [00:22,  1.33s/it]
18it [00:23,  1.34s/it]
19it [00:25,  1.37s/i

20it [00:30,  1.50s/it]
21it [00:32,  1.60s/it]
22it [00:33,  1.53s/it]
23it [00:35,  1.56s/it]
24it [00:36,  1.54s/it]
25it [00:38,  1.55s/it]
26it [00:39,  1.49s/it]
27it [00:41,  1.49s/it]
28it [00:42,  1.53s/it]
29it [00:44,  1.55s/it]
30it [00:45,  1.58s/it]
31it [00:47,  1.55s/it]
32it [00:48,  1.50s/it]
33it [00:50,  1.52s/it]
34it [00:51,  1.49s/it]
35it [00:53,  1.45s/it]
36it [00:54,  1.46s/it]
37it [00:56,  1.47s/it]
38it [00:57,  1.52s/it]
39it [00:59,  1.51s/it]
40it [01:00,  1.50s/it]
41it [01:02,  1.50s/it]
42it [01:03,  1.49s/it]
43it [01:05,  1.52s/it]
44it [01:06,  1.54s/it]
45it [01:08,  1.53s/it]
46it [01:09,  1.54s/it]
47it [01:11,  1.56s/it]
48it [01:13,  1.54s/it]
49it [01:14,  1.51s/it]
50it [01:15,  1.48s/it]
51it [01:17,  1.49s/it]
52it [01:18,  1.51s/it]
53it [01:20,  1.50s/it]
54it [01:21,  1.50s/it]
55it [01:23,  1.55s/it]
56it [01:24,  1.51s/it]
57it [01:26,  1.51s/it]
 16%|██████▌                                  | 4/25 [13:36<1:11:24, 204.00s/it]
0it [00

68it [01:30,  1.35s/it]
69it [01:31,  1.28s/it]
70it [01:32,  1.31s/it]
71it [01:34,  1.31s/it]
72it [01:35,  1.32s/it]
73it [01:36,  1.33s/it]
74it [01:38,  1.32s/it]
75it [01:39,  1.29s/it]
76it [01:40,  1.29s/it]
77it [01:42,  1.32s/it]
78it [01:43,  1.33s/it]
79it [01:44,  1.33s/it]
80it [01:46,  1.34s/it]
81it [01:47,  1.35s/it]
82it [01:48,  1.33s/it]
83it [01:50,  1.37s/it]
84it [01:51,  1.35s/it]
85it [01:52,  1.34s/it]
86it [01:54,  1.33s/it]

0it [00:00, ?it/s]
1it [00:01,  1.50s/it]
2it [00:02,  1.47s/it]
3it [00:04,  1.43s/it]
4it [00:05,  1.44s/it]
5it [00:07,  1.48s/it]
6it [00:08,  1.51s/it]
7it [00:10,  1.54s/it]
8it [00:11,  1.47s/it]
9it [00:13,  1.48s/it]
10it [00:15,  1.54s/it]
11it [00:16,  1.55s/it]
12it [00:18,  1.56s/it]
13it [00:19,  1.58s/it]
14it [00:21,  1.56s/it]
15it [00:22,  1.59s/it]
16it [00:24,  1.50s/it]
17it [00:25,  1.50s/it]
18it [00:27,  1.49s/it]
19it [00:28,  1.45s/it]
20it [00:30,  1.50s/it]
21it [00:32,  1.60s/it]
22it [00:33,  1.52s/it]
23it 

52it [01:09,  1.35s/it]
53it [01:10,  1.32s/it]
54it [01:11,  1.30s/it]
55it [01:13,  1.28s/it]
56it [01:14,  1.32s/it]
57it [01:15,  1.30s/it]
58it [01:17,  1.29s/it]
59it [01:18,  1.32s/it]
60it [01:19,  1.29s/it]
61it [01:20,  1.29s/it]
62it [01:22,  1.31s/it]
63it [01:23,  1.30s/it]
64it [01:24,  1.30s/it]
65it [01:26,  1.33s/it]
66it [01:27,  1.33s/it]
67it [01:28,  1.33s/it]
68it [01:30,  1.34s/it]
69it [01:31,  1.28s/it]
70it [01:32,  1.30s/it]
71it [01:34,  1.30s/it]
72it [01:35,  1.32s/it]
73it [01:36,  1.33s/it]
74it [01:38,  1.32s/it]
75it [01:39,  1.28s/it]
76it [01:40,  1.29s/it]
77it [01:42,  1.32s/it]
78it [01:43,  1.33s/it]
79it [01:44,  1.33s/it]
80it [01:46,  1.34s/it]
81it [01:47,  1.35s/it]
82it [01:48,  1.33s/it]
83it [01:50,  1.37s/it]
84it [01:51,  1.34s/it]
85it [01:52,  1.33s/it]
86it [01:53,  1.32s/it]

0it [00:00, ?it/s]
1it [00:01,  1.50s/it]
2it [00:02,  1.47s/it]
3it [00:04,  1.42s/it]
4it [00:05,  1.44s/it]
5it [00:07,  1.48s/it]
6it [00:08,  1.51s/it]
7i

21it [00:27,  1.35s/it]
22it [00:29,  1.37s/it]
23it [00:30,  1.35s/it]
24it [00:31,  1.32s/it]
25it [00:33,  1.33s/it]
26it [00:34,  1.35s/it]
27it [00:36,  1.34s/it]
28it [00:37,  1.33s/it]
29it [00:38,  1.30s/it]
30it [00:39,  1.31s/it]
31it [00:41,  1.35s/it]
32it [00:42,  1.37s/it]
33it [00:44,  1.34s/it]
34it [00:45,  1.34s/it]
35it [00:46,  1.33s/it]
36it [00:48,  1.35s/it]
37it [00:49,  1.35s/it]
38it [00:50,  1.30s/it]
39it [00:51,  1.29s/it]
40it [00:53,  1.29s/it]
41it [00:54,  1.30s/it]
42it [00:55,  1.36s/it]
43it [00:57,  1.33s/it]
44it [00:58,  1.36s/it]
45it [01:00,  1.36s/it]
46it [01:01,  1.38s/it]
47it [01:02,  1.36s/it]
48it [01:04,  1.33s/it]
49it [01:05,  1.33s/it]
50it [01:06,  1.34s/it]
51it [01:08,  1.34s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:

52it [01:09,  1.35s/it]
53it [01:10,  1.32s/it]
54it [01:11,  1.

50it [01:15,  1.47s/it]
51it [01:17,  1.49s/it]
52it [01:18,  1.51s/it]
53it [01:20,  1.50s/it]
54it [01:21,  1.49s/it]
55it [01:23,  1.55s/it]
56it [01:24,  1.51s/it]
57it [01:25,  1.51s/it]
 68%|████████████████████████████▌             | 17/25 [57:42<27:09, 203.64s/it]
0it [00:00, ?it/s]
1it [00:01,  1.32s/it]
2it [00:02,  1.35s/it]
3it [00:03,  1.30s/it]
4it [00:05,  1.29s/it]
5it [00:06,  1.32s/it]
6it [00:07,  1.32s/it]
7it [00:09,  1.34s/it]
8it [00:10,  1.32s/it]
9it [00:11,  1.32s/it]
10it [00:13,  1.34s/it]
11it [00:14,  1.30s/it]
12it [00:15,  1.29s/it]
13it [00:17,  1.31s/it]
14it [00:18,  1.33s/it]
15it [00:19,  1.35s/it]
16it [00:21,  1.37s/it]
17it [00:22,  1.33s/it]
18it [00:23,  1.33s/it]
19it [00:25,  1.36s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:

20it [00:26,  1.37s/it]
21it [00:27,  1.35s/it]
22it [00:29,  1.37s/i

31it [00:47,  1.54s/it]
32it [00:48,  1.49s/it]
33it [00:50,  1.51s/it]
34it [00:51,  1.48s/it]
35it [00:52,  1.44s/it]
36it [00:54,  1.46s/it]
37it [00:55,  1.47s/it]
38it [00:57,  1.51s/it]
39it [00:58,  1.51s/it]
40it [01:00,  1.49s/it]
41it [01:01,  1.49s/it]
42it [01:03,  1.48s/it]
43it [01:04,  1.51s/it]
44it [01:06,  1.53s/it]
45it [01:08,  1.53s/it]
46it [01:09,  1.54s/it]
47it [01:11,  1.55s/it]
48it [01:12,  1.54s/it]
49it [01:14,  1.50s/it]
50it [01:15,  1.47s/it]
51it [01:17,  1.49s/it]
52it [01:18,  1.51s/it]
53it [01:20,  1.50s/it]
54it [01:21,  1.49s/it]
55it [01:23,  1.55s/it]
56it [01:24,  1.51s/it]
57it [01:25,  1.50s/it]
 76%|██████████████████████████████▍         | 19/25 [1:04:31<20:25, 204.24s/it]
0it [00:00, ?it/s]
1it [00:01,  1.32s/it]
2it [00:02,  1.35s/it]
3it [00:03,  1.30s/it]
4it [00:05,  1.29s/it]
5it [00:06,  1.32s/it]
6it [00:07,  1.32s/it]
7it [00:09,  1.34s/it]
8it [00:10,  1.32s/it]
9it [00:11,  1.32s/it]
10it [00:13,  1.33s/it]
11it [00:14,  1.30s/i

12it [00:18,  1.55s/it]
13it [00:19,  1.58s/it]
14it [00:21,  1.56s/it]
15it [00:22,  1.59s/it]
16it [00:24,  1.50s/it]
17it [00:25,  1.50s/it]
18it [00:27,  1.49s/it]
19it [00:28,  1.45s/it]
20it [00:30,  1.50s/it]
21it [00:31,  1.59s/it]
22it [00:33,  1.52s/it]
23it [00:34,  1.56s/it]
24it [00:36,  1.53s/it]
25it [00:38,  1.55s/it]
26it [00:39,  1.49s/it]
27it [00:40,  1.48s/it]
28it [00:42,  1.53s/it]
29it [00:44,  1.54s/it]
30it [00:45,  1.57s/it]
31it [00:47,  1.54s/it]
32it [00:48,  1.49s/it]
33it [00:50,  1.51s/it]
34it [00:51,  1.49s/it]
35it [00:52,  1.45s/it]
36it [00:54,  1.46s/it]
37it [00:55,  1.47s/it]
38it [00:57,  1.51s/it]
39it [00:58,  1.51s/it]
40it [01:00,  1.50s/it]
41it [01:01,  1.50s/it]
42it [01:03,  1.48s/it]
43it [01:04,  1.51s/it]
44it [01:06,  1.53s/it]
45it [01:08,  1.53s/it]
46it [01:09,  1.54s/it]
47it [01:11,  1.55s/it]
48it [01:12,  1.54s/it]
49it [01:14,  1.50s/it]
50it [01:15,  1.48s/it]
51it [01:17,  1.49s/it]
52it [01:18,  1.51s/it]
53it [01:20,  1.

60it [01:19,  1.29s/it]
61it [01:20,  1.29s/it]
62it [01:22,  1.30s/it]
63it [01:23,  1.30s/it]
64it [01:24,  1.30s/it]
65it [01:26,  1.33s/it]
66it [01:27,  1.33s/it]
67it [01:28,  1.33s/it]
68it [01:30,  1.34s/it]
69it [01:31,  1.28s/it]
70it [01:32,  1.30s/it]
71it [01:33,  1.30s/it]
72it [01:35,  1.32s/it]
73it [01:36,  1.33s/it]
74it [01:38,  1.32s/it]
75it [01:39,  1.29s/it]
76it [01:40,  1.30s/it]
77it [01:41,  1.32s/it]
78it [01:43,  1.33s/it]
79it [01:44,  1.33s/it]
80it [01:45,  1.34s/it]
81it [01:47,  1.35s/it]
82it [01:48,  1.33s/it]
83it [01:50,  1.37s/it]
84it [01:51,  1.34s/it]
85it [01:52,  1.34s/it]
86it [01:53,  1.32s/it]

0it [00:00, ?it/s]
1it [00:01,  1.50s/it]
2it [00:02,  1.46s/it]
3it [00:04,  1.43s/it]
4it [00:05,  1.45s/it]
5it [00:07,  1.48s/it]
6it [00:08,  1.51s/it]
7it [00:10,  1.54s/it]
8it [00:11,  1.47s/it]
9it [00:13,  1.48s/it]
10it [00:15,  1.54s/it]
11it [00:16,  1.55s/it]
12it [00:18,  1.56s/it]
13it [00:19,  1.58s/it]
14it [00:21,  1.56s/it]
15it 

In [12]:
metrics_lst

[{'ptype': 'is_rct',
  'model_type': 'hinge',
  'y_test': 0       0
  1       1
  2       0
  3       0
  4       1
         ..
  7995    0
  7996    1
  7997    0
  7998    1
  7999    1
  Name: is_rct, Length: 8000, dtype: int64,
  'db': -1.584230192290271,
  'df': array([-1.04781593,  3.99362064, -0.12792949, ..., -0.56673838,
          1.47217621,  1.63359905]),
  'auc_roc': 0.91500375,
  'recall_test': 0.985,
  'spec_test': 0.3135,
  'precision_test': 0.11166493754022432,
  'f1_test': 0.20058991531875534},
 {'ptype': 'is_rct',
  'model_type': 'log_loss',
  'y_test': 0       1
  1       0
  2       0
  3       0
  4       1
         ..
  7995    1
  7996    1
  7997    1
  7998    1
  7999    0
  Name: is_rct, Length: 8000, dtype: int64,
  'db': -1.7706254025899768,
  'df': array([ 2.56437989, -2.36902622, -0.9905455 , ...,  2.08079802,
         -0.29060715, -1.12184053]),
  'auc_roc': 0.9198798750000001,
  'recall_test': 0.985,
  'spec_test': 0.36725,
  'precision_test': 0.1200121